# ML Pipeline — Social Media Engagement Analyzer

## 1. Problem framing
Communications and fundraising leadership need to know **which kinds of posts** tend to perform well so they can plan content without guessing. **Stakeholders:** outreach staff, communications manager, executive sponsor for fundraising. **Predictive vs explanatory (textbook):** We use **both**—explanatory models (Ridge/logistic) describe *associations* between post attributes and outcomes; predictive models (with train/test splits) estimate **out-of-sample** performance for ranking and recommendation. Prediction is appropriate when the goal is to **prioritize** future post ideas; explanation supports **interpretability** and stakeholder trust. **Justification:** The business decision is partly “what should we try next?” (prediction) and partly “what seems to matter?” (explanation)—so a dual track matches the decision types in Ch. 1-style framing.

## 2. Data acquisition, preparation & exploration
Data come from **`social_media_posts`** (CSV under `data/lighthouse_csv_v7/`). The notebook loads via `data_prep.load_social_media_posts`, profiles missingness and distributions, and builds modeling frames in `feature_engineering.build_modeling_frame`. **Join logic:** This pipeline is single-table at post grain; optional enrichment from other domains would require explicit keys (not used in the default path). **Feature engineering:** Categorical encoding, time features, text-derived proxies—**excluding outcome leakage** (e.g. likes, impressions as predictors). **Reproducibility:** Python modules (`config`, `preprocess`, `train_*`, `export_artifacts`, `run_all`) implement the repeatable pipeline per Ch. 7—not only ad hoc notebook cells.

## 3. Modeling & feature selection
**Explanatory:** Regularized linear models for interpretable coefficients on scaled features. **Predictive:** Comparable algorithms with **time-ordered or randomized train/test** splits as documented in training scripts. **Feature selection:** Driven by regularization, importance from tree models where used, and domain rules (leakage columns dropped). **Hyperparameters:** Where tuned, search ranges are documented in training code. **Multiple algorithms:** Compared where the notebook and `train_predictive` expose them; final export picks operational models for the app.

## 4. Evaluation & interpretation
Metrics include **RMSE / R² / classification metrics** as appropriate to each target (engagement, referral/donation signals). **Validation:** Holdout or CV as implemented—notebook and `evaluate.py` document splits. **Business interpretation:** Coefficients and importances are translated into “platform/topic/timing tends to associate with higher engagement *holding other modeled factors constant*”—not causal. **Errors:** A **false positive** (predicting a “winner” that underperforms) wastes creative effort; a **false negative** (missing a strong format) leaves engagement on the table. Costs depend on how tightly the team trusts rankings for spend (boost budget).

## 5. Causal and relationship analysis
Feature importances and signs describe **correlational structure in historical posts**, not proven causes (confounding: campaigns, seasonality, audience growth). **Theoretical sense:** e.g. clear CTAs or certain topics may align with fundraising theory, but we **do not** claim causal lift without experiments. **Predictive models** still reveal **stable patterns** useful for ranking even when causal identification is impossible. **Honest limits:** No A/B test inference here; recommendations are **model-assisted**, not guaranteed incremental effects.

## 6. Deployment notes
**Artifacts:** `serialized_models/*.joblib` + metadata under `ml_pipeline_social_media_engagement/`. **Backend bundle:** `python3 refresh_ml_artifacts.py --social-only` or `PYTHONPATH=ml-pipelines python3 -m ml_backend_export.run_all_backend_exports --social-only` writes to `backend/Intex.API/App_Data/ml/social/`. **Live inference:** FastAPI `ml_service/main.py` loads `SocialRecommenderSession` from `ml_pipeline_social_media_engagement.recommend_posts`; .NET proxies via `POST /api/ml/social/recommend`. **Admin UI:** `frontend/src/pages/scaffold/SocialMediaSuggestPage.tsx`, “Best next post” widget on `AdminHomePage.tsx`.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

def _find_repo_root(start: Path) -> Path:
    p = start.resolve()
    for d in [p, *p.parents]:
        if (d / "ml-pipelines").is_dir() and (d / "data" / "lighthouse_csv_v7").is_dir():
            return d
    raise FileNotFoundError(
        "Could not find repo root (need ml-pipelines/ and data/lighthouse_csv_v7/). "
        f"cwd={p}"
    )

REPO_ROOT = _find_repo_root(Path.cwd())
ML_PIPELINES_DIR = REPO_ROOT / "ml-pipelines"
ML_PIPELINE = ML_PIPELINES_DIR
if str(ML_PIPELINES_DIR) not in sys.path:
    sys.path.insert(0, str(ML_PIPELINES_DIR))

from ml_pipeline_social_media_engagement import config
from ml_pipeline_social_media_engagement.data_prep import load_social_media_posts, profile_raw_frame
from ml_pipeline_social_media_engagement.feature_engineering import LEAKAGE_COLUMNS, build_modeling_frame

sns.set_theme(style="whitegrid")
print("ML_PIPELINE:", ML_PIPELINE)
print("DATA:", config.DEFAULT_SOCIAL_CSV)

## Step 1 — Inspect raw data
Row count, dtypes, missingness, and category coverage.

In [ ]:
raw = load_social_media_posts()
prof = profile_raw_frame(raw)
print("n_rows:", prof["n_rows"], "n_columns:", prof["n_columns"])
print("columns:", prof["columns"])

In [ ]:
miss = raw.isna().mean().sort_values(ascending=False)
display(miss.to_frame("missing_frac").head(20))

fig, ax = plt.subplots(figsize=(8, 5))
miss.head(18).iloc[::-1].plot(kind="barh", ax=ax, color="steelblue")
ax.set_title("Highest missingness (raw)")
plt.tight_layout()
config.OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
plt.savefig(config.OUTPUTS_DIR / "nb_missingness.png", dpi=120)
plt.show()

In [ ]:
for col in ["platform", "post_type", "media_type", "content_topic", "sentiment_tone"]:
    if col in raw.columns:
        print(f"\n--- {col} (top 12) ---")
        print(raw[col].value_counts(dropna=False).head(12))

In [ ]:
raw["created_at"] = pd.to_datetime(raw["created_at"], errors="coerce")
print("created_at range:", raw["created_at"].min(), "→", raw["created_at"].max())
raw["post_hour"] = pd.to_numeric(raw["post_hour"], errors="coerce")
fig, ax = plt.subplots(figsize=(8, 3))
raw["post_hour"].dropna().astype(int).value_counts().sort_index().plot(kind="bar", ax=ax, color="seagreen")
ax.set_title("Post hour distribution")
plt.tight_layout()
plt.show()

### Targets (distributions)
`engagement_rate` is the primary engagement target. `donation_referrals` is skewed; `estimated_donation_value_php` has a long tail.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
raw["engagement_rate"].hist(bins=40, ax=axes[0], color="coral")
axes[0].set_title("engagement_rate")
raw["donation_referrals"].clip(upper=raw["donation_referrals"].quantile(0.95)).hist(bins=40, ax=axes[1], color="teal")
axes[1].set_title("donation_referrals (capped 95p display)")
np.log1p(raw["estimated_donation_value_php"].clip(lower=0)).hist(bins=40, ax=axes[2], color="slateblue")
axes[2].set_title("log1p(estimated_donation_value_php)")
plt.tight_layout()
plt.show()
print("P(any referral):", (raw["donation_referrals"] > 0).mean())

## Step 2 — Leakage-safe modeling frame
Columns in `LEAKAGE_COLUMNS` are **not** used as predictors for the main strategy models.

In [ ]:
print("Excluded as X (leakage / outcomes):", LEAKAGE_COLUMNS)
df, meta = build_modeling_frame(raw)
print("Modeled rows:", len(df))
print("Numeric features:", meta["numeric_features"])
print("Categorical features:", meta["categorical_features"])

## Step 3 — EDA on targets by key categories
Grouped summaries (associational).

In [ ]:
tmp = raw.dropna(subset=["engagement_rate", "platform"])
order = tmp.groupby("platform")["engagement_rate"].median().sort_values().index.tolist()
fig, ax = plt.subplots(figsize=(8, 4))
sns.boxplot(data=tmp, x="engagement_rate", y="platform", order=order, ax=ax)
ax.set_title("engagement_rate by platform")
plt.tight_layout()
plt.show()

In [ ]:
g = (
    raw.groupby("post_type", dropna=False)
    .agg(
        median_engagement=("engagement_rate", "median"),
        mean_referrals=("donation_referrals", "mean"),
        n=("post_id", "count"),
    )
    .sort_values("median_engagement", ascending=False)
)
display(g.head(15))

In [ ]:
sns.barplot(
    data=raw,
    x="features_resident_story",
    y="engagement_rate",
    errorbar="pi",
)
plt.title("engagement_rate vs resident story flag (raw)")
plt.tight_layout()
plt.show()

sns.barplot(
    data=raw,
    x="has_call_to_action",
    y="donation_referrals",
    errorbar="pi",
)
plt.title("donation_referrals vs CTA present (raw)")
plt.tight_layout()
plt.show()

## Step 4 — Train pipelines & export artifacts
Runs explanatory models, predictive comparisons, saves `joblib` + `social_media_engagement_metadata.json` + charts under `outputs/` and `serialized_models/`.

In [ ]:
from ml_pipeline_social_media_engagement.export_artifacts import export_all

info = export_all()
print(json.dumps(info, indent=2, default=str))

In [ ]:
with open(config.SERIALIZED_DIR / "social_media_engagement_metadata.json", encoding="utf-8") as f:
    m = json.load(f)
print("Holdout engagement best:", m["predictive_holdout"]["engagement"]["best_model"], m["predictive_holdout"]["engagement"]["metrics"])
print("Holdout referrals count best:", m["predictive_holdout"]["donation_referrals_count"]["best_model"], m["predictive_holdout"]["donation_referrals_count"]["metrics"])
print("Any referral classifier best:", m["predictive_holdout"]["any_referral"]["best_model"], m["predictive_holdout"]["any_referral"]["metrics"])

## Step 5 — Interpretation (explanatory coefficients)
Read CSVs in `outputs/`. Positive coef → higher feature level **associated with** higher target (after scaling / one-hot), holding other linear terms fixed — **not** a causal claim.

In [ ]:
coef_eng = pd.read_csv(config.OUTPUTS_DIR / "explanatory_ridge_engagement_rate.csv")
display(coef_eng.head(20))
coef_ref = pd.read_csv(config.OUTPUTS_DIR / "explanatory_ridge_donation_referrals.csv")
display(coef_ref.head(20))

## Step 6 — Optional: secondary “post-performance” lens
**Only for exploration:** you may regress `engagement_rate` on `likes`, `comments`, `click_throughs`, etc. to describe *correlation of outcomes with each other*. **Do not** treat that as a planning model for future posts (those metrics are unknown before the post runs).

```python
# Example only — not used in export_artifacts
# y = raw['engagement_rate']
# X_perf = raw[['likes','comments','shares','impressions']].fillna(0)
# ... fit with clear label: POST_PERFORMANCE_EXPLORATORY
```

## Summary checklist
1. **Columns used:** see `meta` from `build_modeling_frame` + `social_media_engagement_metadata.json`.
2. **Targets:** `engagement_rate`, `donation_referrals`, `log1p(estimated_donation_value_php)`, binary `referrals_positive` / `referrals_high`.
3. **Dropped:** free text `caption` / `hashtags` string, IDs, `campaign_name` (sparse), all leakage columns listed in README.
4. **Best models:** in metadata JSON under `predictive_holdout`.
5. **Insights:** combine grouped EDA + coefficient tables; phrase as associations.
6. **Next steps:** time-based validation, more data, calibration, hierarchical models for platform-specific effects.